# Convergence population — array-stack driver (cardio-only)

<details>
<summary>Replicates My-ICU-Twin's [`Convergence_Population.ipynb`](https://github.com/mtc8608/My-ICU-Twin/blob/main/Convergence_Population.ipynb) on this repo's array-mode stack.</summary>

**The study.** Draw `nrModels` random parameter sets (Latin Hypercube) over the cardiovascular calibration parameters, then for each set run a *staged calibration* (`runner.runCalibration` walks `calibration.stages` with an escalating `multiplierC` — this repo's equivalent of the reference's “double the cubic factor each loop”) toward one fixed physiological **twin** target state. Measure how close each model converges, building an error distribution across the population.

All runs are written to a manuBeat-compatible **population artifact** via `hdf5/schema_pop.py` (`final_states` table + `runs/{id}/raw` traces), the same file a downstream NN calibration (`schema_calib.build_training_set`) would consume.

Config lives in `config/scenarios/sepsis.json`: `shared.twin` (targets + volume distribution), `calibration.stages` (the convergence schedule), and `convergence` (parameter bounds + observation list).

**Integration.** Runs on the step-independent (SI) stack via `runner.run` one sample at a time, so the solver (`euler`/`rk4`) and precision (float64/float32) are selectable in the imports/config cells — the serial counterpart to the batched `Convergence_Run_Batch.ipynb`.

</details>

In [ ]:
# region -> runConfig — the single run-configuration surface (device/precision applied before JAX)
# The ONE place run configuration lives (repo CLAUDE.md); defined first so device/precision applies before JAX.
runConfig = {
    # --- file references ---
    "model":    "cvModel.json",   # cardio-only model (16 calibration controllers)
    "scenario": "sepsis.json",    # shared.twin + calibration.stages + convergence
    "mode":     "calibration",    # each member is a staged calibration run

    # --- pipeline phases ---
    "run":  True,   # Phase 1 — LHS sweep + save
    "plot": True,   # Phase 2 — load + analyse + plot

    # --- device / precision (applied before `import jax`) ---
    "device": {
        "useGpu":    False,       # CPU (True -> CUDA)
        "precision": "float64",   # "float64" (parity) or "float32"
    },

    # --- integration stack + solver ---
    "stack":     "SI",              # SI stack honours the solver below (legacy is Euler-only)
    "solver":    {"type": "euler"}, # SI fixed-step solver: "euler"/"rk4"

    # --- calibration overrides ---
    "calibration": {                # per-key overrides of scenario calibration ({} = as-is)
    },

    # --- population sweep ---
    "population": {
        "nrModels":    1,         # sweep size (start small, then scale)
        "seed":        0,         # LHS reproducibility
        "errorTarget": 0.5,       # SUCCESS if max |rel err| (%) <= this
    },

    # --- analysis ---
    "analysis": {
        "atm": 760.0,               # atmospheric offset (gauge = raw - atm)
        "divergenceLimit": 2000.0,  # |value| >= this in any obs/param -> BAD run
    },

    # --- output ---
    "output": {"save": True, "path": "data/convergence", "name": "population_hr_test.h5",
               "logProgress": True},   # persist live-progress trace into the .h5

    # --- post-processing (off: population file is the artifact) ---
    "postProcessing": None,
    "requested":      None,
    "plots":          [],

    # --- printing ---
    "printStatus":    True,       # per-run integrator timing lines
    "printEveryPct":  10,         # print run progress every this % of nrModels
    "progressEvery":  0,          # within-sample convergence line every N sim-seconds (0 = off)

    # --- integration numerics (override scenario shared.integration) ---
    "runTime": 10,       # simulated seconds per internal run
    "dt":      0.0005,   # integrator step
    "dtDense": 1.0,      # save/output grid: 1.0 = 1 Hz; raise for within-beat waveforms
}
# endregion

## Imports

In [ ]:
# region -> imports + device/precision (must precede `import jax`)
# ---- repo-root bootstrap: run from any cwd (make `library` importable + resolve
# ---- the relative data/ + config/ paths). Walks up to the dir containing library/. ----
import os, sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "library")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)

# ---- device / precision (from runConfig, MUST run before JAX initialises) -------
useGpu    = runConfig["device"]["useGpu"]
precision = runConfig["device"]["precision"]

import os
if useGpu:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)
    os.environ["JAX_PLATFORMS"] = "cuda"
    # GPU memory hygiene (must precede `import jax`): grow on demand instead of grabbing
    # ~75% of VRAM up front (so JAX coexists with the display on a small shared card), and
    # hand freed buffers back to the driver so the cleanup cell / del actually releases VRAM.
    os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
    os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]   = "platform"
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
    os.environ["JAX_PLATFORMS"] = "cpu"

import jax
jax.config.update("jax_enable_x64", precision == "float64")

import library.run.runner as runner
import library.run.stateSetup as stateSetup   # resolveCalibrationBounds (single-source calibration ranges)
import library.run.progress as progressLib   # standard live-progress line + record log            # run orchestration (modes + stage stacks)
import library.viz.plots as libPlots           # plotCalibrationConvergence
import library.utils as utils
from library.hdf5 import schema_pop                        # population artifact (init_population / add_run)
import library.postproc.reporting as reporting             # shared scope / rejection report
import numpy as np
from scipy.stats import qmc                        # Latin Hypercube sampling (SALib not required)
import pandas as pd
import matplotlib.pyplot as plt
import json
import time

np.set_printoptions(suppress=True)
print("devices:", jax.devices(), "| x64:", jax.config.jax_enable_x64)
# endregion

## Assemble `simulationParams` + load the convergence config

<details>
<summary>`runner.buildSimulationParams` expands the slim `runConfig` + scenario into the legacy</summary>

`simulationParams` shape. The `convergence` block of the scenario carries the parameter
bounds (LHS) and the observation list (the controllers' `varTarget`s + `V_Vs`).

</details>

In [ ]:
# region -> assemble simulationParams + load the convergence config (params, bounds, observations)
scenario = utils.loadScenario(runConfig["scenario"])
simulationParams = runner.buildSimulationParams(runConfig, scenario)

model         = utils.loadJSONfile(utils.configPath("models", runConfig["model"]))
conv          = scenario["convergence"]
calConf       = simulationParams["simulationConf"]["calibration"]
twin          = scenario["shared"]["twin"]["twinTargets"]
volDist       = scenario["shared"]["twin"]["volumeDistribution"]
param_names   = calConf["adaptive"]["parameters"]                                    # swept set = calibration params
bounds        = stateSetup.resolveCalibrationBounds(model["calibration"], calConf, param_names)  # (P, 2) single source
observations  = conv["observations"]
pop           = runConfig["population"]
outPath       = os.path.join(runConfig["output"]["path"], runConfig["output"]["name"])

print(f"nrModels = {pop['nrModels']} | params = {len(param_names)} | observations = {len(observations)}")
print(f"output -> {outPath}")
# endregion

## Twin-target array + atmospheric offsets

<details>
<summary>Each observation has a target derived from the twin state: pressures from the twin pressure</summary>

targets, stroke volume from `CO/HR`, compartment volumes from `volumeDistribution ×
TotalBloodVolume`, and `Cyc_HC` from `60/HR`. Absolute-pressure observations carry the model's
+760 mmHg atmospheric offset, which we subtract to compare in gauge mmHg (amplitudes, volumes
and SV carry no offset).

</details>

In [ ]:
# region -> twin-target array + atmospheric offsets per observation
ATM = runConfig["analysis"]["atm"]

def obsOffset(name):
    """Atmospheric offset baked into absolute-pressure signals (gauge = raw - offset)."""
    return utils.obsOffset(name, ATM)

def obsTarget(name):
    """Twin target for an observation (gauge units), or None if untargeted.

    Matches Convergence_Population.ipynb's targetsArray: the capillary means are
    derived as a pressure DROP from the upstream arterial target, not as absolute
    means -- avg_P_Cs = (Sys_P_As - amp_P_As) - twin.avg_P_Cs (systemic diastolic
    minus the configured drop), avg_P_Cp = Dia_P_Ap - twin.avg_P_Cp.
    """
    TBV = twin["TotalBloodVolume"]
    direct = {
        "avg_P_Vs": twin["CVP"],
        "avg_P_Cs": twin["Sys_P_As"] - twin["amp_P_As"] - twin["avg_P_Cs"],
        "avg_P_Cp": twin["Dia_P_Ap"] - twin["avg_P_Cp"],
        "keep_max_P_As": twin["Sys_P_As"], "keep_max_P_Ap": twin["Sys_P_Ap"],
        "keep_min_P_Ap": twin["Dia_P_Ap"], "amp_P_As": twin["amp_P_As"],
        "keep_SV_Hl": twin["CO"] / twin["HR"], "Cyc_HC": 60.0 / twin["HR"],
        "V_Vs": volDist["Vs"] * TBV,
    }
    if name in direct:
        return direct[name]
    if name.startswith("avg_V_"):
        return volDist[name[len("avg_V_"):]] * TBV
    return None

targetArr = np.array([obsTarget(o) if obsTarget(o) is not None else np.nan for o in observations])
offsetArr = np.array([obsOffset(o) for o in observations])
pd.DataFrame({"observation": observations, "target": targetArr, "offset": offsetArr})
# endregion

## Latin Hypercube sample of the parameter space

<details>
<summary>`sampled_params` is an `(nrModels, P)` matrix scaled into each parameter's `[min, max]`</summary>

bounds. Each row becomes the initial value of the corresponding controlled state for one
population member (`runner.run(..., stateOverrides=row)`).

</details>

In [ ]:
# region -> Latin Hypercube sample of the parameter space
if runConfig["run"]:
    sampler = qmc.LatinHypercube(d=len(param_names), seed=pop["seed"])
    unit = sampler.random(n=pop["nrModels"])                      # (N, P) in [0, 1)
    sampled_params = qmc.scale(unit, bounds[:, 0], bounds[:, 1])   # (N, P) in [min, max]
    print("sampled_params shape:", sampled_params.shape)
    pd.DataFrame(sampled_params, columns=param_names).head()
# endregion

## Run the population

<details>
<summary>For each sample: inject the draw as `stateOverrides`, run the staged calibration, read each</summary>

observation's steady-state value (last completed-cycle value) from `results`, and append the
run to the population file. `schema_pop.add_run` sanitizes any NaN / diverging run to
`BAD_RUN_SENTINEL`; solver exceptions are caught and recorded as a sentinel run so the sweep
never aborts. The population file is initialized lazily from the first run so its `state_names`
and `model_structure` exactly match the runtime.

</details>

In [ ]:
# region -> run the population: staged calibration per LHS sample, save to population artifact
if runConfig["run"]:
    def steadyState(results, name):
        """Gauge steady-state value of an observation (last value minus atmospheric offset)."""
        return utils.steadyState(results, name, ATM)

    if runConfig["output"]["save"]:
        os.makedirs(runConfig["output"]["path"], exist_ok=True)

    obsMatrix = np.full((pop["nrModels"], len(observations)), np.nan)  # in-memory mirror for plots
    rawSignals = list(dict.fromkeys(list(observations) + list(param_names)))  # obs + swept params -> raw
    runWall = np.full(pop["nrModels"], np.nan)                        # per-run wall clock (s), persisted
    initialized = False
    # print run progress every this % of nrModels (last run always prints)
    printEvery = max(1, round(pop["nrModels"] * runConfig.get("printEveryPct", 10) / 100))
    # Standard live-progress line: obs-space |rel err| vs twin targets over the samples completed
    # so far (serial's analogue of the batched population line). Logged for later comparison.
    reporter = progressLib.ProgressReporter(logEnabled=runConfig["output"].get("logProgress", True))
    t0 = time.time()

    for i in range(pop["nrModels"]):
        override = {p: float(sampled_params[i, j]) for j, p in enumerate(param_names)}
        tRun = time.time()
        try:
            states, modelObjects, modelStructure, results, structures = runner.run(
                simulationParams, stateOverrides=override)
            rawTraces = {n: np.asarray(results[n]) for n in rawSignals if n in results}
            obsMatrix[i] = [steadyState(results, o) if o in results else np.nan for o in observations]
            finalState = states
        except Exception as e:
            print(f"  [run {i}] solver exception: {e}")
            rawTraces = {n: np.array([schema_pop.BAD_RUN_SENTINEL]) for n in rawSignals}
            finalState = {p: schema_pop.BAD_RUN_SENTINEL for p in param_names}
        runWall[i] = time.time() - tRun

        if runConfig["output"]["save"]:
            if not initialized:
                schema_pop.init_population(
                    outPath,
                    param_names=param_names,
                    state_names=list(finalState.keys()),
                    observation_names=observations,
                    sampled_params=sampled_params,
                    model_structure=utils.modelStructureJSON(modelStructure),
                    problem={"names": param_names, "bounds": bounds.tolist(),
                             "num_vars": len(param_names)},
                    conf=runConfig, meta={"twinTargets": twin})
                initialized = True
            schema_pop.add_run(outPath, run_id=str(i), raw=rawTraces, final_state=finalState)

        if (i + 1) % printEvery == 0 or i == pop["nrModels"] - 1:
            done = i + 1
            reporter.emit(kind="step", label=f"run {done}/{pop['nrModels']}",
                          done=done, total=pop["nrModels"], elapsedWall=time.time() - t0,
                          stats=progressLib.relErrorStats(obsMatrix[:done], targetArr))

    if runConfig["output"]["save"]:
        schema_pop.write_timings(outPath, runWall, meta={
            "device":    "gpu" if useGpu else "cpu",
            "precision": precision,
            "solver":    simulationParams["solver"]["type"],
            "dt":        simulationParams["dt"],
            "runTime":   simulationParams["runTime"],
            "nrModels":  pop["nrModels"],
            "stack":     runConfig["stack"],
            "total_wall": time.time() - t0,
        })
        # persist the live-progress trace (the convergence lines printed above) for comparison
        schema_pop.write_progress(outPath, reporter.records, meta={
            "model": runConfig["model"], "scenario": runConfig["scenario"],
            "mode": runConfig["mode"], "solver": simulationParams["solver"]["type"],
            "nrModels": pop["nrModels"], "stack": runConfig["stack"]})

    print(f"Population complete in {time.time() - t0:.0f}s -> {outPath}")
# endregion

# Phase 2 — Load & analyse (from the saved file)

<details>
<summary>Everything below reconstructs from the population `.h5` — no in-session run state</summary>

is required. After a kernel restart you can run the setup cells (config →
`simulationParams` → targets → LHS, all scenario-only and cheap) then the **load**
cell below and every analysis/plot/table cell, without re-running the sweep.

The load cell rebuilds `modelStructure` (from the stored `model_structure` JSON),
`obsMatrix` (each observation's steady-state value = last saved raw value minus
its atmospheric offset, in `sampled_params` order), and the per-run timings
(`runWall`, `timingMeta`) written in Phase 1.

</details>

In [ ]:
# region -> load everything the analysis needs straight from the saved population file
if runConfig["plot"]:
    # --- load everything the analysis below needs, straight from the saved file ----------
    # Reconstructs the in-memory run state (modelStructure, obsMatrix) + per-run timings so
    # Phase 2 is independent of Phase 1 having run in this kernel.
    import h5py

    with h5py.File(outPath, "r") as f:
        modelStructure = json.loads(f["model_structure"].asstr()[()])
        runIds = list(f["run_ids"].asstr()[:])
        obsMatrix = np.full((len(runIds), len(observations)), np.nan)
        paramMatrix = np.full((len(runIds), len(param_names)), np.nan)   # run-end swept/calibrated params
        for rid in runIds:
            i = int(rid)
            grp = f[f"runs/{rid}/raw"]
            for j, o in enumerate(observations):
                if o in grp:
                    obsMatrix[i, j] = float(np.asarray(grp[o]).flat[-1]) - obsOffset(o)
            for j, p in enumerate(param_names):        # a run whose parameter leaves scope is BAD too
                if p in grp:
                    paramMatrix[i, j] = float(np.asarray(grp[p]).flat[-1])
    paramInScope = list(param_names)

    runWall, timingMeta = schema_pop.read_timings(outPath)
    print(f"loaded {obsMatrix.shape[0]} runs from {outPath} | timings: {np.isfinite(runWall).sum()} timed")
# endregion

## Scope / rejection report

<details>
<summary>Breaks down which runs were dropped and **why**, before the error summary works on the survivors.</summary>

A run is out of scope when **any** observation *or* swept/calibrated parameter reaches `|value| >=
divergenceLimit` (`runConfig.analysis.divergenceLimit`), or the lane is NaN/Inf or sentinel-stamped.
Reports the total kept/dropped split, the per-reason counts (a run can trip more than one), and the
individual signals that drove the out-of-scope drops with their worst run-end magnitude.

</details>

In [ ]:
# region -> scope / rejection report (why each run was dropped)
if runConfig["plot"]:
    lim = runConfig["analysis"].get("divergenceLimit", 1e6)
    # obsMatrix is un-blanked here (serial does not NaN-mask it in load), so pass it as rawObs.
    rep = reporting.scopeRejectionReport(obsMatrix, paramMatrix, observations, paramInScope, lim=lim)
# endregion

## Error summary

<details>
<summary>Filter faulty runs (any observation == `BAD_RUN_SENTINEL`), then compute per-observation</summary>

relative error `(obs - target) / target × 100` and absolute error against the twin targets.
A run is SUCCESS when `max |relative error| <= errorTarget`.

</details>

In [ ]:
# region -> error summary — drop faulty runs, per-observation relative/absolute error
if runConfig["plot"]:
    targeted = ~np.isnan(targetArr)                     # observations that have a twin target
    good = schema_pop.good_run_mask(obsMatrix, runConfig["analysis"].get("divergenceLimit", 1e6),
                                    param_matrix=paramMatrix)
    print(f"{good.sum()}/{len(good)} valid runs ({(~good).sum()} faulty/diverged dropped)")

    obsGood = obsMatrix[good][:, targeted]
    tgt = targetArr[targeted]
    errRel = (obsGood - tgt) / tgt * 100.0
    errAbs = obsGood - tgt
    obsTargeted = [o for o, t in zip(observations, targeted) if t]

    success = np.max(np.abs(errRel), axis=1) <= pop["errorTarget"]
    print(f"converged (max|rel err| <= {pop['errorTarget']}%): {success.sum()}/{len(success)}")

    summary = pd.DataFrame({
        "observation": obsTargeted,
        "target": tgt,
        "mean_rel_err_%": np.nanmean(errRel, axis=0),
        "std_rel_err_%": np.nanstd(errRel, axis=0),
        "mean_abs_err": np.nanmean(errAbs, axis=0),
    })
    summary
# endregion

## Plots

In [ ]:
# region -> boxplot: relative error distribution per observation
if runConfig["plot"]:
    # --- boxplot: relative error distribution per observation ---------------------------
    if good.sum() > 0:
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.boxplot(errRel, tick_labels=utils.labelsFor(obsTargeted, "latex"), showfliers=False)
        ax.axhline(0.0, color="k", lw=0.8)
        ax.axhline(pop["errorTarget"], color="r", ls="--", lw=0.8, label=f"±{pop['errorTarget']}% target")
        ax.axhline(-pop["errorTarget"], color="r", ls="--", lw=0.8)
        ax.set_ylabel("relative error (%)")
        ax.set_title(f"Convergence error across {good.sum()} valid runs")
        ax.tick_params(axis="x", rotation=90)
        ax.legend()
        plt.tight_layout()
        plt.show()
# endregion

## LaTeX summary table

<details>
<summary>Builds the Table-5-style summary (each targeted observation paired with the parameter that</summary>

controls it), rendered with paper-quality names from `config/labels.json` via
`utils.labelFor`, and written to `notebookData/convergence_summary.tex`.

</details>

In [ ]:
# region -> Table-5-style LaTeX summary
if runConfig["plot"]:
    # --- Table-5-style LaTeX summary -----------------------------------------------------
    # Each targeted observation paired with the calibration parameter that controls it
    # (controller varTarget -> param), with paper-quality names from config/labels.json.
    calib = modelStructure["calibration"]
    paramForObs = {calib[p]["params"]["varTarget"]: p
                   for p in param_names if p in calib}        # observation -> controlling param

    arrFS, stateNames = schema_pop.final_states_array(outPath)  # (N, S) calibrated final states
    sIdx = {n: i for i, n in enumerate(stateNames)}
    paramVals = {p: arrFS[good, sIdx[p]] for p in param_names if p in sIdx}

    rows = {}
    for j, o in enumerate(obsTargeted):
        p = paramForObs.get(o, "")
        pv = paramVals.get(p)
        has_p = pv is not None and pv.size > 0
        rows[utils.labelFor(o, "latex")] = [
            f"{tgt[j]:.4g}",
            f"{summary['mean_rel_err_%'][j]:.3g}",
            f"{summary['std_rel_err_%'][j]:.2g}",
            utils.labelFor(p, "latex") if p else "--",
            f"{np.mean(pv):.4g}" if has_p else "--",
            f"{np.std(pv):.2g}" if has_p else "--",
        ]

    latexTable = utils.generate_latex_table_new(
        rows,
        ["Vars", "Targets", "Rel. Error \\%", "std", "Param", "Value", "std"],
        "", "", "convergence",
        f"Summary of the convergence test across {good.sum()} simulations with random initial "
        f"parameter values and volume distributions. For each calibration target, the table "
        f"reports the prescribed target value, the mean signed relative error and its standard "
        f"deviation at convergence, together with the corresponding calibrated parameter values.")

    out_tex = os.path.join(runConfig["output"]["path"], "convergence_summary.tex")
    with open(out_tex, "w") as f:
        f.write(latexTable)
    print(latexTable)
    print(f"\nLaTeX table written to: {out_tex}")
# endregion

## Calibration convergence plot

<details>
<summary>Port of the legacy `buildConvPlot`: one panel per **targeted observation**, overlaying the</summary>

per-run observation trace (one jet-colored line per converged population member, read back
from the HDF5 `runs/{id}/raw/<obs>` datasets) against its twin target (dashed). Shows how
the population converges toward each target across the saved calibration window.

</details>

In [ ]:
# region -> calibration convergence: per-run observation traces vs target
if runConfig["plot"]:
    # --- calibration convergence: per-run observation traces vs target -------------------
    if runConfig["output"]["save"]:
        import h5py
        targetsByObs = {o: t for o, t in zip(observations, targetArr)}
        offsetsByObs = {o: off for o, off in zip(observations, offsetArr)}
        # observation -> controlling calibration parameter (for the panel titles)
        paramForObsPlot = {calib[p]["params"]["varTarget"]: p for p in param_names if p in calib}

        goodTraces = {}
        with h5py.File(outPath, "r") as f:
            for o in obsTargeted:
                rows = []
                for rid in f["runs"]:
                    tr = f[f"runs/{rid}/raw/{o}"][()] if o in f[f"runs/{rid}/raw"] else np.array([])
                    if tr.size > 1 and tr.flat[-1] > schema_pop.BAD_RUN_SENTINEL + 1.0:
                        rows.append(np.asarray(tr, dtype=float))
                # keep only signals whose good-run traces share a common length (stackable)
                if rows and all(len(r) == len(rows[0]) for r in rows):
                    goodTraces[o] = np.vstack(rows)

        libPlots.plotCalibrationConvergence(
            goodTraces, traceT=None, targets=targetsByObs, offsets=offsetsByObs,
            paramForObs=paramForObsPlot,
            title="Calibration convergence (serial)")
        plt.show()
# endregion

## Parameter convergence plot

<details>
<summary>Companion to the observation plot above, for the **swept calibration parameters**. One panel</summary>

per parameter, overlaying each converged member's parameter trajectory (jet) with the **LHS
sampling edges** `[min, max]` as dotted lines — showing how tightly the population settles
inside the range it was sampled from. No target line (parameters have no fixed target) and no
legend.

</details>

In [ ]:
# region -> calibration convergence: one panel per swept parameter
if runConfig["plot"]:
    # --- calibration convergence: one panel per swept parameter --------------------------------
    # Each panel: the per-run parameter trajectory (jet) and the LHS sampling edges [min, max]
    # (dotted grey). No legend, no target line (parameters have no fixed target).
    if runConfig["output"]["save"]:
        import h5py
        rangesByParam = {p: tuple(bounds[j]) for j, p in enumerate(param_names)}          # edges

        paramTraces = {}
        with h5py.File(outPath, "r") as f:
            for p in param_names:
                rows = []
                for rid in f["runs"]:
                    grp = f[f"runs/{rid}/raw"]
                    tr = grp[p][()] if p in grp else np.array([])
                    if tr.size > 1 and tr.flat[-1] > schema_pop.BAD_RUN_SENTINEL + 1.0:
                        rows.append(np.asarray(tr, dtype=float))
                # keep only params whose good-run traces share a common length (stackable)
                if rows and all(len(r) == len(rows[0]) for r in rows):
                    paramTraces[p] = np.vstack(rows)

        libPlots.plotCalibrationConvergence(
            paramTraces, traceT=None, ranges=rangesByParam,
            paramForObs=None, showLegend=False,
            title="Calibration convergence -- parameters (serial)")
        plt.show()
# endregion

## Run timings

<details>
<summary>Per-calibration wall-clock cost, loaded from the population file (`runWall`,</summary>

`timingMeta`). Directly supports the reviewer's timing ask (wall-clock per
calibration): the aggregate table reports mean/median/min/max seconds per run and
the total for this device/precision/solver, and `libPlots.plotRunTimings` shows
the per-run series and its distribution. Cross-file CPU-vs-GPU and scaling-vs-N
comparisons live in the dedicated timing-analysis notebook.

</details>

In [ ]:
# region -> run timing summary table + plot
if runConfig["plot"]:
    # --- timing summary table + plot -----------------------------------------------------
    wallFinite = runWall[np.isfinite(runWall)]
    if wallFinite.size > 0:
        timingSummary = pd.DataFrame([{
            "device":     timingMeta.get("device", "?"),
            "precision":  timingMeta.get("precision", "?"),
            "solver":     timingMeta.get("solver", "?"),
            "dt":         timingMeta.get("dt"),
            "runTime":    timingMeta.get("runTime"),
            "nrModels":   int(wallFinite.size),
            "mean_s/run":   float(np.mean(wallFinite)),
            "median_s/run": float(np.median(wallFinite)),
            "min_s/run":    float(np.min(wallFinite)),
            "max_s/run":    float(np.max(wallFinite)),
            "sum_run_s":    float(np.sum(wallFinite)),
            "total_wall_s": timingMeta.get("total_wall"),
        }])
        display(timingSummary)
        libPlots.plotRunTimings(runWall, meta=timingMeta,
                                title="Calibration wall-clock (serial)")
        plt.show()
    else:
        print("no finite timings to summarise")
# endregion

## (Deferred) NN training set

<details>
<summary>The population file is the input to the inverse-problem calibration NN (X = observations,</summary>

y = parameters). Deferred this round; the hook is:

```python
from library.hdf5 import schema_pop, schema_calib
arr, state_names = schema_pop.final_states_array(outPath)
training = schema_calib.build_training_set(
    arr, state_names, observation_keys=observations, param_keys=param_names)
schema_calib.save_calibration_run("notebookData/calibration_hr_test.h5", **training)
```

Note: this needs the observation values stored as columns in `final_states`; the current run
loop stores observations only as `runs/{id}/raw` traces. Wire the steady-state observation
values into the `final_state` row (or a parallel table) before enabling this.

</details>

In [ ]:
# region -> release GPU memory
# ---- release GPU memory --------------------------------------------------------------
# Drop references to the big result arrays, clear JAX's compiled caches, and (with
# XLA_PYTHON_CLIENT_ALLOCATOR=platform from the device cell) hand the VRAM back to the
# driver -- no kernel restart needed. A kernel restart is still the guaranteed full reset.
import gc
for _v in ("batch", "results", "finalStates", "traces", "traceT", "obsMatrix",
           "prep", "layout", "sampled_params", "finiteRow", "fsArr", "goodTraces", "block",
           "runWall"):
    globals().pop(_v, None)
jax.clear_caches()
gc.collect()
try:
    used = sum((d.memory_stats() or {}).get("bytes_in_use", 0) for d in jax.devices())
    print(f"GPU bytes in use after cleanup: {used/1e6:.0f} MB (restart kernel for a full reset)")
except Exception as e:
    print("memory_stats() unavailable on this device:", e)
# endregion